# Train a DINOv2 particle classifier

End-to-end training pipeline for a particle classifier built on a frozen DINOv2-small vision backbone and a trainable linear head.  The exported model is a self-contained TorchScript file that drops directly into a PyOPIA pipeline via `pyopia.classify_torch.Classify`.

**Pipeline outline:**
1. Load a labelled dataset (PyOPIA's example SilCam database, or your own).
2. Split into train / validation with a fixed seed.
3. Each training epoch, re-extract training features under a random augmentation policy (rotations, intensity/colour jitter, blur) and update a linear head with AdamW.
4. Track train loss, validation loss and validation macro-F1; keep the best-F1 head.
5. Export backbone + head as a single traced `.pt` with a `[0, 255]` input contract that matches PyOPIA's preprocessing.
6. Round-trip-verify the exported model against the in-notebook head.

**Your own dataset:**
1. Images in folders labeled by the class names
2. Set USE_EXAMPLE_DATABASE = False and provide the path to your image dataset (DATA_DIR)

**Augmentations applied at training time:**
- Random continuous rotation (0–360°) with white fill for corner regions.
- Random intensity jitter (brightness + contrast).
- Random colour jitter (saturation + hue).
- Random Gaussian blur.

Augmentation strength is controlled by a single scalar (`AUGMENT_STRENGTH`) in the configuration cell.

**Input contract:**
- Training pipeline converts images to `[0, 1]` float tensors (`ToTensor()`).
- The exported `.pt` accepts `[0, 255]` inputs and bakes `/255` into its forward pass, matching `pyopia.classify_torch.Classify`'s convention.

**Compute:** one backbone forward pass per training image per epoch.  GPU or MPS recommended; CPU is workable but slow.

## Install extra dependencies
In addition to PyOPIA (for testing at the end), PyTorch, this notebook requires [TIMM](https://pypi.org/project/timm/) and scikit-learn. 

In [ ]:
!uv pip install -q torch timm scikit-learn

## Configuration

In [ ]:
# ════════════════ EDIT THIS SECTION ════════════════

# Option A: PyOPIA example database (downloads automatically)
USE_EXAMPLE_DATABASE = True

# Option B: your own labelled data (set USE_EXAMPLE_DATABASE = False)
DATA_DIR = ""

# Training images per class (remaining images used for validation)
IMAGES_PER_CLASS = 50

# Training epochs.  20–50 is typical; more rarely helps.
EPOCHS = 30

# Mini-batch size for per-epoch feature extraction.  Larger = faster,
# bounded by GPU memory.  64–256 works on most GPUs.
BATCH_SIZE = 64

# Number of augmented views to extract per training image per epoch.
# 1 = standard per-epoch augmentation.  Higher = stronger regularisation
# at proportional extra cost.  Leave at 1 unless you have spare compute.
VIEWS_PER_EPOCH = 1

# Augmentation strength (multiplier on jitter/blur intensities).
# 0.0 = rotation only.  1.0 = standard.  2.0 = aggressive.
AUGMENT_STRENGTH = 1.0

# Output path for the exported model
OUTPUT_PATH = "model_augmented.pt"

# ════════════════════════════════════════════════==

## Download / validate dataset

In [ ]:
from pathlib import Path
import numpy as np

if USE_EXAMPLE_DATABASE:
    import pyopia.exampledata
    DATA_DIR = pyopia.exampledata.get_classifier_database_from_pysilcam_blob(
        download_directory="silcam_classification_database"
    )
    print(f"Using PyOPIA example database: {DATA_DIR}")

data_path = Path(DATA_DIR)
assert data_path.is_dir(), f"Data directory not found: {DATA_DIR}"

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp"}

classes = sorted([d.name for d in data_path.iterdir() if d.is_dir()])
class_to_idx = {name: i for i, name in enumerate(classes)}
num_classes = len(classes)

all_samples = []
for cls_name in classes:
    cls_dir = data_path / cls_name
    imgs = sorted(
        p for p in cls_dir.iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    )
    for p in imgs:
        all_samples.append((p, class_to_idx[cls_name]))

print(f"\nFound {len(all_samples)} images across {num_classes} classes:")
for cls_name in classes:
    count = sum(1 for _, lbl in all_samples if lbl == class_to_idx[cls_name])
    print(f"  {cls_name}: {count} images")

## Train / validation split

In [ ]:
rng = np.random.default_rng(42)

train_samples = []
val_samples = []

for cls_idx in range(num_classes):
    cls_samples = [s for s in all_samples if s[1] == cls_idx]
    indices = np.arange(len(cls_samples))
    rng.shuffle(indices)
    n_train = min(IMAGES_PER_CLASS, len(cls_samples) - 1)
    for i in indices[:n_train]:
        train_samples.append(cls_samples[i])
    for i in indices[n_train:]:
        val_samples.append(cls_samples[i])
    if n_train < IMAGES_PER_CLASS:
        print(f"  Warning: {classes[cls_idx]} has only {len(cls_samples)} images "
              f"(using {n_train} for training)")

print(f"\nTraining: {len(train_samples)} images")
print(f"Validation: {len(val_samples)} images")

## Load DINOv2 backbone and define transforms

Both transforms output `[0, 1]` tensors via `ToTensor()`.  The linear head absorbs any constant offset, and skipping normalisation keeps the training / inference input contract trivial.

In [ ]:
import torch
import torch.nn as nn
import timm
from PIL import Image
from torchvision import transforms
from tqdm.auto import tqdm

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Device: {device}")

IMG_SIZE = 224

backbone = timm.create_model(
    "vit_small_patch14_dinov2.lvd142m",
    pretrained=True,
    num_classes=0,
    img_size=IMG_SIZE,
    dynamic_img_size=False,
)
backbone.eval()
backbone.requires_grad_(False)
backbone = backbone.to(device)

num_prefix_tokens = int(getattr(backbone, "num_prefix_tokens", 1))
embed_dim = backbone.embed_dim
feature_dim = 2 * embed_dim

s = AUGMENT_STRENGTH
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(
        degrees=360,
        interpolation=transforms.InterpolationMode.BILINEAR,
        fill=255,
    ),
    transforms.ColorJitter(
        brightness=0.3 * s,
        contrast=0.3 * s,
        saturation=0.2 * s,
        hue=0.05 * s,
    ),
    transforms.RandomApply(
        [transforms.GaussianBlur(kernel_size=5, sigma=(0.1, 1.5))],
        p=0.3 * s,
    ),
    transforms.ToTensor(),  # -> [0, 1]
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),  # -> [0, 1]
])

print(f"Backbone: DINOv2-s ({sum(p.numel() for p in backbone.parameters()):,} params)")
print(f"Feature dim: {feature_dim} (CLS + mean patches)")

## Pre-load training images into RAM

Disk I/O dominates per-epoch extraction otherwise.  We keep raw PIL
images in memory and re-apply the augmentation pipeline each epoch.

In [ ]:
train_pil = []
y_train = []
for path, label in tqdm(train_samples, desc="Loading train images"):
    train_pil.append(Image.open(path).convert("RGB").copy())
    y_train.append(label)
y_train = torch.tensor(y_train, dtype=torch.long)
print(f"Loaded {len(train_pil)} training images into RAM")

## Extract validation features once (deterministic)

In [ ]:
@torch.no_grad()
def extract_features_batched(pil_images, transform_fn, desc="Extracting"):
    """Extract probe features (CLS + mean patches) in mini-batches."""
    features = []
    for i in tqdm(range(0, len(pil_images), BATCH_SIZE), desc=desc):
        batch = pil_images[i:i + BATCH_SIZE]
        x = torch.stack([transform_fn(img) for img in batch]).to(device)
        tokens = backbone.forward_features(x)
        cls = tokens[:, 0]
        patches = tokens[:, num_prefix_tokens:].mean(dim=1)
        feat = torch.cat([cls, patches], dim=-1)
        features.append(feat.cpu())
    return torch.cat(features, dim=0)


val_pil = [Image.open(p).convert("RGB") for p, _ in tqdm(val_samples, desc="Loading val images")]
y_val = torch.tensor([lbl for _, lbl in val_samples], dtype=torch.long)
X_val = extract_features_batched(val_pil, eval_transform, "Extracting val features")
print(f"\nVal features: {X_val.shape}")

## Train linear head with per-epoch augmented features

Each epoch: re-extract training features under random augmentation, shuffle, run mini-batch AdamW updates on the head, then evaluate on the cached validation features.  The best-F1 head is kept at the end.

In [ ]:
from sklearn.metrics import f1_score

head = nn.Linear(feature_dim, num_classes)
optimizer = torch.optim.AdamW(head.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_f1 = -1.0
best_state = None
best_epoch = 0
history = {"epoch": [], "train_loss": [], "val_loss": [], "val_macro_f1": [], "lr": []}

for epoch in range(1, EPOCHS + 1):
    feat_chunks = []
    label_chunks = []
    for _ in range(VIEWS_PER_EPOCH):
        chunk = extract_features_batched(
            train_pil, train_transform,
            desc=f"Epoch {epoch}/{EPOCHS} extract",
        )
        feat_chunks.append(chunk)
        label_chunks.append(y_train)
    X_epoch = torch.cat(feat_chunks, dim=0)
    y_epoch = torch.cat(label_chunks, dim=0)

    head.train()
    perm = torch.randperm(len(X_epoch))
    epoch_loss = 0.0
    n_batches = 0
    for i in range(0, len(X_epoch), BATCH_SIZE):
        idx = perm[i:i + BATCH_SIZE]
        logits = head(X_epoch[idx])
        loss = criterion(logits, y_epoch[idx])
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        n_batches += 1
    scheduler.step()
    avg_loss = epoch_loss / max(n_batches, 1)

    head.eval()
    with torch.no_grad():
        val_logits = head(X_val)
        val_loss = criterion(val_logits, y_val).item()
        val_preds = val_logits.argmax(dim=1).numpy()
    macro_f1 = f1_score(y_val.numpy(), val_preds, average="macro", zero_division=0)

    history["epoch"].append(epoch)
    history["train_loss"].append(avg_loss)
    history["val_loss"].append(val_loss)
    history["val_macro_f1"].append(macro_f1)
    history["lr"].append(scheduler.get_last_lr()[0])

    if macro_f1 > best_f1:
        best_f1 = macro_f1
        best_epoch = epoch
        best_state = {k: v.clone() for k, v in head.state_dict().items()}

    print(f"  Epoch {epoch:3d}/{EPOCHS}  train_loss={avg_loss:.4f}  "
          f"val_loss={val_loss:.4f}  val_macro_f1={macro_f1:.3f}  "
          f"lr={scheduler.get_last_lr()[0]:.2e}")

head.load_state_dict(best_state)
head.eval()
print(f"\nBest val macro-F1: {best_f1:.3f} (epoch {best_epoch})")

## Training curves

Train vs. validation loss, plus validation macro-F1 over epochs. A growing gap between train and val loss — or val loss turning up while train loss keeps falling — is the usual overfitting signature.

In [ ]:
import matplotlib.pyplot as plt

epochs_axis = history["epoch"]

fig, (ax_loss, ax_f1) = plt.subplots(1, 2, figsize=(11, 4))

ax_loss.plot(epochs_axis, history["train_loss"], label="train loss", marker="o", markersize=3)
ax_loss.plot(epochs_axis, history["val_loss"], label="val loss", marker="o", markersize=3)
ax_loss.axvline(best_epoch, color="grey", linestyle="--", alpha=0.6, label=f"best epoch ({best_epoch})")
ax_loss.set_xlabel("epoch")
ax_loss.set_ylabel("cross-entropy loss")
ax_loss.set_title("Training vs. validation loss")
ax_loss.legend()
ax_loss.grid(True, alpha=0.3)

ax_f1.plot(epochs_axis, history["val_macro_f1"], color="tab:green", marker="o", markersize=3, label="val macro-F1")
ax_f1.axvline(best_epoch, color="grey", linestyle="--", alpha=0.6, label=f"best epoch ({best_epoch})")
ax_f1.axhline(best_f1, color="tab:green", linestyle=":", alpha=0.5, label=f"best F1 = {best_f1:.3f}")
ax_f1.set_xlabel("epoch")
ax_f1.set_ylabel("macro-F1")
ax_f1.set_title("Validation macro-F1")
ax_f1.set_ylim(0, 1)
ax_f1.legend()
ax_f1.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Evaluate

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

with torch.no_grad():
    val_preds = head(X_val).argmax(dim=1).numpy()
y_true = y_val.numpy()

print(classification_report(y_true, val_preds, target_names=classes, zero_division=0))

cm = confusion_matrix(y_true, val_preds, normalize="true")
fig, ax = plt.subplots(figsize=(max(num_classes * 0.8, 5), max(num_classes * 0.7, 4)))
ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=classes,
).plot(ax=ax, cmap="Blues", values_format=".2f")
ax.set_title(f"Validation confusion matrix (macro-F1 = {best_f1:.3f})")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## Export for PyOPIA

Wraps backbone + head with a `[0, 255]` -> `[0, 1]` rescale baked in.
The exported `.pt` file is a self-contained inference contract that
matches `pyopia.classify_torch.Classify`'s preprocessing convention
(which mirrors the TF-Keras `pyopia.classify` exactly).

In [ ]:
import json
import hashlib


class ClassifierWrapper(nn.Module):
    """Backbone + head with [0, 255] -> [0, 1] rescaling baked in."""

    def __init__(self, backbone, head, num_prefix):
        super().__init__()
        self.backbone = backbone
        self.head = head
        self.num_prefix = num_prefix

    def forward(self, x):  # x in [0, 255]
        x = x / 255.0
        tokens = self.backbone.forward_features(x)
        cls = tokens[:, 0]
        patches = tokens[:, self.num_prefix:].mean(dim=1)
        features = torch.cat([cls, patches], dim=-1)
        return self.head(features)


wrapper = ClassifierWrapper(backbone.cpu(), head.cpu(), num_prefix_tokens)
wrapper.eval()

# Trace with a [0, 255] example to match the runtime contract.
example_input = torch.rand(1, 3, IMG_SIZE, IMG_SIZE) * 255.0
traced = torch.jit.trace(wrapper, example_input)

metadata = {
    "class_labels": classes,
    "img_height": IMG_SIZE,
    "img_width": IMG_SIZE,
}

torch.jit.save(
    traced,
    OUTPUT_PATH,
    _extra_files={"metadata.json": json.dumps(metadata)},
)

size_mb = Path(OUTPUT_PATH).stat().st_size / (1024 * 1024)
with open(OUTPUT_PATH, "rb") as f:
    model_hash = hashlib.file_digest(f, "sha256").hexdigest()[:16]

print(f"Saved {OUTPUT_PATH} ({size_mb:.1f} MB, {num_classes} classes, "
      f"macro-F1 = {best_f1:.3f}, hash = {model_hash}...)")
print(f"\nAdd this to your PyOPIA config.toml:\n")
print(f"    [steps.classifier]")
print(f"    pipeline_class = 'pyopia.classify_torch.Classify'")
print(f"    model_path = '{OUTPUT_PATH}'")

## Round-trip sanity check

Load the exported model via PyOPIA's `Classify` and compare its predictions to the in-notebook head on the validation set.  Agreement should be high; residual disagreement reflects minor preprocessing differences (PIL vs skimage resize, hardware float numerics) and is concentrated on samples near decision boundaries.

In [ ]:
from pyopia.classify_torch import Classify

cl = Classify(
    model_path=OUTPUT_PATH,
    normalize_intensity=False,
    correct_whitebalance=False,
)

pyopia_preds = []
for path, _ in tqdm(val_samples, desc="PyOPIA inference"):
    img = Image.open(path).convert("RGB")
    roi = np.array(img, dtype=np.float64) / 255.0
    probs = cl.proc_predict(roi)
    pyopia_preds.append(int(np.argmax(probs)))
pyopia_preds = np.array(pyopia_preds)

pyopia_macro_f1 = f1_score(y_true, pyopia_preds, average="macro", zero_division=0)
agreement = float((pyopia_preds == val_preds).mean())
print(f"PyOPIA round-trip macro-F1 : {pyopia_macro_f1:.3f}")
print(f"Notebook macro-F1          : {best_f1:.3f}")
print(f"Agreement with notebook    : {agreement:.1%} "
      f"({(pyopia_preds == val_preds).sum()}/{len(y_true)})")
if agreement < 0.99:
    print("  ⚠ Lower than expected — check preprocessing pipeline.")

cm_pyopia = confusion_matrix(y_true, pyopia_preds, normalize="true")
fig, ax = plt.subplots(figsize=(max(num_classes * 0.8, 5), max(num_classes * 0.7, 4)))
ConfusionMatrixDisplay(
    confusion_matrix=cm_pyopia,
    display_labels=classes,
).plot(ax=ax, cmap="Purples", values_format=".2f")
ax.set_title(f"PyOPIA round-trip confusion matrix (macro-F1 = {pyopia_macro_f1:.3f})")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## Robustness check: same particle under random augmentations

In [ ]:
NUM_PARTICLES = num_classes  # one row per class
NUM_AUGMENTATIONS = 5         # extra columns (original + N augmentations)

# PIL-only augmentation (Classify does ToTensor / scaling internally).
aug_pil = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(
        degrees=360,
        interpolation=transforms.InterpolationMode.BILINEAR,
        fill=255,
    ),
    transforms.ColorJitter(
        brightness=0.3 * AUGMENT_STRENGTH,
        contrast=0.3 * AUGMENT_STRENGTH,
        saturation=0.2 * AUGMENT_STRENGTH,
        hue=0.05 * AUGMENT_STRENGTH,
    ),
    transforms.RandomApply(
        [transforms.GaussianBlur(kernel_size=5, sigma=(0.1, 1.5))],
        p=0.3 * AUGMENT_STRENGTH,
    ),
])

def classify_pil(img):
    roi = np.array(img.convert("RGB"), dtype=np.float64) / 255.0
    probs = cl.proc_predict(roi)
    idx = int(np.argmax(probs))
    return classes[idx], float(probs[idx])

# Pick one representative validation particle per class.
chosen = []
seen = set()
for i in rng.permutation(len(val_samples)):
    p, lbl = val_samples[i]
    if lbl not in seen:
        chosen.append((p, lbl))
        seen.add(lbl)
    if len(chosen) == NUM_PARTICLES:
        break

ncols = 1 + NUM_AUGMENTATIONS
fig, axes = plt.subplots(
    NUM_PARTICLES, ncols,
    figsize=(2.2 * ncols, 2.6 * NUM_PARTICLES),
    squeeze=False,
)

for row, (path, true_label) in enumerate(chosen):
    true_class = classes[true_label]
    base_img = Image.open(path).convert("RGB")

    pred_class, conf = classify_pil(base_img)
    ax = axes[row][0]
    ax.imshow(base_img)
    color = "green" if pred_class == true_class else "red"
    ax.set_title(
        f"original\ntrue: {true_class}\npred: {pred_class} ({conf:.0%})",
        color=color, fontsize=8,
    )
    ax.axis("off")

    for col in range(1, ncols):
        aug_img = aug_pil(base_img)
        pred_class, conf = classify_pil(aug_img)
        ax = axes[row][col]
        ax.imshow(aug_img)
        color = "green" if pred_class == true_class else "red"
        ax.set_title(
            f"aug #{col}\npred: {pred_class} ({conf:.0%})",
            color=color, fontsize=8,
        )
        ax.axis("off")

plt.suptitle(
    "Robustness check — one particle per class under random augmentations\n"
    "(green = correct, red = incorrect)",
    fontsize=10,
)
plt.tight_layout()
plt.show()

---

**Done.** The exported `.pt` is ready to drop into your PyOPIA pipeline via the `config.toml` snippet printed above.